Genetic Algo with Roulette Wheel Selection, Single Point Crossover

In [ ]:
import random

def calculate_objective_and_fitness(population, verbose=False):
    """Steps 1 & 2: Calculate Objective Value and Fitness"""
    objective_values = []
    fitness_values = []
    total_fitness = 0
    
    if verbose: print("\n--- STEP 1 & 2: OBJECTIVE & FITNESS ---")
    
    for i, chrom in enumerate(population):
        a, b, c, d = chrom
        # Objective Function: val = (a + 2*b + 3*c + 4*d) - 30
        val = (a + 2*b + 3*c + 4*d) - 30
        objective_values.append(val)
        
        # Fitness Function: 1.0 / (1.0 + error)
        # We assume optimal 'val' is 0, so error is abs(val)
        error = abs(val)
        fitness = 1.0 / (1.0 + error)
        fitness_values.append(fitness)
        total_fitness += fitness
        
        if verbose: print(f"Chrom {i} {chrom}: Val={val}, Error={error}, Fitness={fitness:.4f}")
            
    return fitness_values, total_fitness

def selection_step(population, fitness_values, total_fitness, verbose=False):
    """Steps 3-6: Probabilities and Roulette Wheel Selection"""
    # Step 3: Probabilities
    probabilities = [f / total_fitness for f in fitness_values]
    
    # Step 4: Cumulative Probabilities
    cumulative_probs = []
    current_cum = 0.0
    for prob in probabilities:
        current_cum += prob
        cumulative_probs.append(current_cum)
    
    # Steps 5 & 6: Selection
    new_population = []
    if verbose: print("\n--- STEP 5 & 6: SELECTION ---")
    
    # Generate random numbers for selection
    random_numbers = [random.random() for _ in range(len(population))]
    
    for r in random_numbers:
        selected_index = 0
        prev_cum = 0.0
        for i, cum in enumerate(cumulative_probs):
            if r <= cum:
                selected_index = i
                if verbose: print(f"Random {r:.4f}: {prev_cum:.4f} < r <= {cum:.4f} -> Select Chrom {i}")
                break
            prev_cum = cum
        new_population.append(population[selected_index])
        
    return new_population

def crossover_step(population, crossover_rate=0.25, verbose=False):
    """Steps 7-11: Crossover"""
    if verbose: print("\n--- STEP 7-11: CROSSOVER ---")
    
    parents_indices = []
    
    # Step 10 loop: Ensure we don't select exactly 1 parent (needs pairs)
    while True:
        crossover_randoms = [random.random() for _ in range(len(population))]
        parents_indices = [i for i, r in enumerate(crossover_randoms) if r <= crossover_rate]
        
        if len(parents_indices) != 1:
            break
        if verbose: print("Only 1 parent selected. Regenerating randoms...")

    # Step 11: Single Point Crossover
    new_population = population[:] # Copy population
    
    # Process pairs
    for i in range(0, len(parents_indices) - 1, 2):
        p1_idx = parents_indices[i]
        p2_idx = parents_indices[i+1]
        
        parent1 = new_population[p1_idx]
        parent2 = new_population[p2_idx]
        
        # Random split point between index 1 and 3 (assuming length 4)
        point = random.randint(1, 3)
        
        child1 = parent1[:point] + parent2[point:]
        child2 = parent2[:point] + parent1[point:]
        
        new_population[p1_idx] = child1
        new_population[p2_idx] = child2
        
        if verbose: print(f"Crossover {p1_idx}&{p2_idx} at {point}: {child1}, {child2}")
            
    return new_population

def mutation_step(population, mutation_rate=0.10, verbose=False):
    """Step 12: Mutation"""
    if verbose: print("\n--- STEP 12: MUTATION ---")
    
    for i in range(len(population)):
        for j in range(len(population[i])):
            r = random.random()
            if r < mutation_rate:
                old_val = population[i][j]
                # Generating new random gene (assuming 0-30 range)
                new_val = random.randint(0, 30) 
                population[i][j] = new_val
                if verbose: print(f"Mutated Chrom {i} gene {j}: {old_val} -> {new_val}")
                
    return population

# --- Main Reusable Function ---

def run_genetic_algorithm(iterations=100, verbose=False):
    # Initial Data
    current_population = [
        [12, 5, 23, 8],
        [2, 21, 18, 3],
        [10, 4, 13, 14],
        [20, 1, 10, 6],
        [1, 4, 13, 19],
        [20, 5, 17, 1]
    ]

    best_solution = None
    best_fitness_global = -1

    print(f"Starting Genetic Algorithm for {iterations} iterations...")
    if not verbose:
        print("(Verbose mode is OFF. Showing summary per generation.)")

    for gen in range(1, iterations + 1):
        if verbose: print(f"\n================ GENERATION {gen} ================")
        
        # 1. Calc Fitness
        fitness_vals, total_fit = calculate_objective_and_fitness(current_population, verbose)
        
        # Check for best solution in this generation
        max_fit = max(fitness_vals)
        best_idx = fitness_vals.index(max_fit)
        
        if max_fit > best_fitness_global:
            best_fitness_global = max_fit
            best_solution = current_population[best_idx]
        
        # Print simple summary if not verbose
        if not verbose:
            print(f"Gen {gen}: Best Fitness = {max_fit:.4f} | Pop Best: {current_population[best_idx]}")
        
        # Stop if perfect solution found (Fitness = 1.0 means Error = 0)
        if best_fitness_global >= 1.0:
            print(f"\nTarget Reached at Generation {gen}!")
            break

        # 2. Selection
        selected_pop = selection_step(current_population, fitness_vals, total_fit, verbose)
        
        # 3. Crossover
        crossover_pop = crossover_step(selected_pop, crossover_rate=0.25, verbose=verbose)
        
        # 4. Mutation
        mutated_pop = mutation_step(crossover_pop, mutation_rate=0.10, verbose=verbose)
        
        # Update population for next loop
        current_population = mutated_pop

    print("\n--- END OF RUN ---")
    print(f"Best Solution Found: {best_solution}")
    val = (best_solution[0] + 2*best_solution[1] + 3*best_solution[2] + 4*best_solution[3]) - 30
    print(f"Objective Value (Target 0): {val}")
    print(f"Fitness Score: {best_fitness_global:.6f}")

# --- Execute ---
# Run for 100 iterations
run_genetic_algorithm(iterations=100, verbose=False)

Starting Genetic Algorithm for 100 iterations...

================ GENERATION 1 ================

--- STEP 1 & 2: OBJECTIVE & FITNESS ---
Chrom 0 [12, 5, 23, 8]: Val=93, Error=93, Fitness=0.0106
Chrom 1 [2, 21, 18, 3]: Val=80, Error=80, Fitness=0.0123
Chrom 2 [10, 4, 13, 14]: Val=83, Error=83, Fitness=0.0119
Chrom 3 [20, 1, 10, 6]: Val=46, Error=46, Fitness=0.0213
Chrom 4 [1, 4, 13, 19]: Val=94, Error=94, Fitness=0.0105
Chrom 5 [20, 5, 17, 1]: Val=55, Error=55, Fitness=0.0179

--- STEP 5 & 6: SELECTION ---
Random 0.3733: 0.2718 < r <= 0.4126 -> Select Chrom 2
Random 0.2969: 0.2718 < r <= 0.4126 -> Select Chrom 2
Random 0.5098: 0.4126 < r <= 0.6643 -> Select Chrom 3
Random 0.1802: 0.1258 < r <= 0.2718 -> Select Chrom 1
Random 0.1840: 0.1258 < r <= 0.2718 -> Select Chrom 1
Random 0.7036: 0.6643 < r <= 0.7888 -> Select Chrom 4

--- STEP 7-11: CROSSOVER ---
Only 1 parent selected. Regenerating randoms...
Only 1 parent selected. Regenerating randoms...
Only 1 parent selected. Regenerating r

GA using pygad library for diabetes dataset 

In [2]:
import pygad
import numpy as np
from sklearn.datasets import load_diabetes
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

# 1. Load Data (Diabetes Medical Dataset)
data = load_diabetes()
X = data.data
y = data.target
feature_names = data.feature_names

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# 2. Define Fitness Function for Feature Selection
def fitness_func_fs(ga_instance, solution, solution_idx):
    # 'solution' is a binary array ( [1, 0, 1, ...])
    # Select features where gene is 1
    selected_indices = [i for i, gene in enumerate(solution) if gene == 1]
    
    # If no features selected, return 0 fitness
    if len(selected_indices) == 0:
        return 0
        
    X_train_sel = X_train[:, selected_indices]
    X_test_sel = X_test[:, selected_indices]
    
    # Train Model
    model = LinearRegression()
    model.fit(X_train_sel, y_train)
    predictions = model.predict(X_test_sel)
    
    # Calculate Error
    mse = mean_squared_error(y_test, predictions)
    
    # Fitness = Inverse of MSE (Minimize Error -> Maximize Fitness)
    return 1.0 / (mse + 1e-5)

# 3. Configure PyGAD for Binary Optimization
ga_fs = pygad.GA(
    num_generations=20,
    num_parents_mating=4,
    fitness_func=fitness_func_fs,
    sol_per_pop=10,
    num_genes=len(feature_names), 
    gene_type=int,                
    init_range_low=0,
    init_range_high=2,            
    parent_selection_type="tournament", 
    crossover_type="single_point",
    mutation_type="random",
    mutation_probability=0.1
)

# 4. Run
print("\nRunning GA for Feature Selection...")
ga_fs.run()

# 5. Results
best_sol, best_fit, best_idx = ga_fs.best_solution()
selected_features = [feature_names[i] for i, gene in enumerate(best_sol) if gene == 1]

print(f"\n--- Part B Result ---")
print(f"Original Features: {len(feature_names)}")
print(f"Selected Features: {len(selected_features)}")
print(f"Feature List: {selected_features}")
print(f"Best MSE (Error): {1.0/best_fit:.2f}")


Running GA for Feature Selection...

--- Part B Result ---
Original Features: 10
Selected Features: 5
Feature List: ['bmi', 's1', 's2', 's5', 's6']
Best MSE (Error): 2764.09


GA using NiaPy library for minimizing f(x) = (a + 2b + 3c + 4d)

In [3]:
import numpy as np
from niapy.problems import Problem
from niapy.task import Task
from niapy.algorithms.basic import GeneticAlgorithm

# 1. Define the Custom Problem Class
class EquationProblem(Problem):
    def __init__(self, dimension=4, lower=1, upper=30):
        # We have 4 variables (a, b, c, d)
        super().__init__(dimension=dimension, lower=lower, upper=upper)
        self.weights = np.array([1, 2, 3, 4])
        self.target = 30

    def _evaluate(self, x):
        # NiaPy passes 'x' as a numpy array of floats
        # We round them to integers for this specific integer-like problem
        params = np.round(x)
        
        # Calculate: (a + 2b + 3c + 4d)
        result = np.sum(params * self.weights)
        
        # We want to minimize the difference from 30
        error = abs(result - self.target)
        return error

# 2. Setup the Task
# We define the optimization task (Minimization is default)
task = Task(problem=EquationProblem(), max_evals=10000)

# 3. Configure and Run Genetic Algorithm
# NiaPy handles population initialization internally
algo = GeneticAlgorithm(population_size=100, crossover_probability=0.8, mutation_probability=0.2)
best_x, best_fit = algo.run(task)

# 4. Results
print("--- Part A: NiaPy Equation Solver ---")
print(f"Best Solution (a, b, c, d): {np.round(best_x)}")
print(f"Final Error: {best_fit} (Target 0.0)")

--- Part A: NiaPy Equation Solver ---
Best Solution (a, b, c, d): [6. 8. 1. 1.]
Final Error: 1.0 (Target 0.0)
